* 허깅페이스: 잘 훈련된 모델을 다운받아 훈련시킬 수 있음

In [ ]:
# !pip install transformers
# !pip install sentencepiece

In [2]:
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer

ko_text = "이것은 m2m모델로 만든 다국어 번역기입니다"
chinese_text = "生活就像一盒巧克力。"

model = M2M100ForConditionalGeneration.from_pretrained("facebook/m2m100_418M")
tokenizer = M2M100Tokenizer.from_pretrained("facebook/m2m100_418M")

# 한국어를 영어로
tokenizer.src_lang = "ko" # 어떤 언어 입력받을 건지
encoded_hi = tokenizer(ko_text, return_tensors="pt")
generated_tokens = model.generate(**encoded_hi, forced_bos_token_id=tokenizer.get_lang_id("en")) # 어떤 언어 출력할건지
result1=tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
print(result1)
# => "La vie est comme une boîte de chocolat."

# 한국어를 일본어로
tokenizer.src_lang = "ko"
encoded_zh = tokenizer(ko_text, return_tensors="pt")
generated_tokens = model.generate(**encoded_zh, forced_bos_token_id=tokenizer.get_lang_id("ja"))
result2=tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
print(result2)


['This is a multi-language translator made with m2m model.']
['これはm2mモデルで作られた多言語翻訳機です。']


In [3]:
import gradio as gr
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer

# 모델 및 토크나이저 로드
model_name = "facebook/m2m100_418M"
model = M2M100ForConditionalGeneration.from_pretrained(model_name)
tokenizer = M2M100Tokenizer.from_pretrained(model_name)

# 지원 언어 목록
lang_dict = {
    "한국어": "ko",
    "영어": "en",
    "중국어": "zh",
    "일본어": "ja",
    "프랑스어": "fr",
    "독일어": "de",
    "스페인어": "es",
    "이탈리아어": "it",
    "포르투갈어": "pt"
}

def translate(text, src_lang, tgt_lang):
    if not text.strip():
        return ""
    
    tokenizer.src_lang = lang_dict[src_lang]
    encoded = tokenizer(text, return_tensors="pt")
    generated_tokens = model.generate(
        **encoded,
        forced_bos_token_id=tokenizer.get_lang_id(lang_dict[tgt_lang])
    )
    return tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]

# Gradio UI 구성
with gr.Blocks() as demo:
    gr.Markdown("## 🌍 다국어 번역기 (Powered by M2M100)")

    with gr.Row():
        src_lang = gr.Dropdown(choices=list(lang_dict.keys()), value="한국어", label="입력 언어")
        tgt_lang = gr.Dropdown(choices=list(lang_dict.keys()), value="영어", label="번역 언어")

    with gr.Row():
        input_text = gr.Textbox(lines=5, label="원본 텍스트")
        output_text = gr.Textbox(lines=5, label="번역 결과")

    translate_button = gr.Button("번역하기")

    translate_button.click(fn=translate, inputs=[input_text, src_lang, tgt_lang], outputs=output_text)

demo.launch()


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [4]:
demo.close()

Closing server running on port: 7860


In [6]:
!pip install datasets soundfile

   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 12.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/25.8 MB ? eta -:--:--
   ------- -------------------------------- 4.7/25.8 MB 23.7 MB/s eta 0:00:01
   ---------------- ----------------------- 10.7/25.8 MB 26.8 MB/s eta 0:00:01
   ----------------------------- ---------- 19.1/25.8 MB 30.9 MB/s eta 0:00:01
   ---------------------------------------  25.7/25.8 MB 33.9 MB/s eta 0:00:01
   ---------------------------------------- 25.8/25.8 MB 28.1 MB/s eta 0:00:00

   -- -------------------------------------  1/16 [pycparser]
   -- -------------------------------------  1/16 [pycparser]
   -- -------------------------------------  1/16 [pycparser]
   -- -------------------------------------  1/16 [pycparser]
   ----- ----------------------------------  2/16 [pyarrow]
   ----- ----------------------------------  2/16 [pyarrow]
   ----- ------------

In [7]:
import gradio as gr
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer
from transformers import SpeechT5Processor, SpeechT5ForTextToSpeech, SpeechT5HifiGan
from datasets import load_dataset
import torch
import soundfile as sf

# 번역 모델 세팅
translation_model = M2M100ForConditionalGeneration.from_pretrained("facebook/m2m100_418M")
translation_tokenizer = M2M100Tokenizer.from_pretrained("facebook/m2m100_418M")

# 음성합성 모델 세팅
tts_processor = SpeechT5Processor.from_pretrained("microsoft/speecht5_tts")
tts_model = SpeechT5ForTextToSpeech.from_pretrained("microsoft/speecht5_tts")
tts_vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan")
xvector_dataset = load_dataset("Matthijs/cmu-arctic-xvectors", split="validation")
speaker_embedding = torch.tensor(xvector_dataset[7306]["xvector"]).unsqueeze(0)

# 언어 코드 매핑
lang_dict = {
    "한국어": "ko",
    "영어": "en",
    "중국어": "zh",
    "일본어": "ja",
    "프랑스어": "fr",
    "독일어": "de",
    "스페인어": "es",
    "이탈리아어": "it",
    "포르투갈어": "pt"
}

# 번역 + TTS 함수
def translate_and_speak(text, src_lang, tgt_lang):
    if not text.strip():
        return "", None

    # 번역
    translation_tokenizer.src_lang = lang_dict[src_lang]
    encoded = translation_tokenizer(text, return_tensors="pt")
    generated_tokens = translation_model.generate(
        **encoded,
        forced_bos_token_id=translation_tokenizer.get_lang_id(lang_dict[tgt_lang])
    )
    translated = translation_tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]

    # 영어일 경우 TTS
    audio_path = None
    if lang_dict[tgt_lang] == "en":
        tts_inputs = tts_processor(text=translated, return_tensors="pt")
        speech = tts_model.generate_speech(tts_inputs["input_ids"], speaker_embedding, vocoder=tts_vocoder)
        audio_path = "speech.wav"
        sf.write(audio_path, speech.numpy(), samplerate=16000)

    return translated, audio_path

# Gradio UI
with gr.Blocks() as demo:
    gr.Markdown("## 🌐 다국어 번역기 + 영어 음성 출력")

    with gr.Row():
        src_lang = gr.Dropdown(choices=list(lang_dict.keys()), value="한국어", label="입력 언어")
        tgt_lang = gr.Dropdown(choices=list(lang_dict.keys()), value="영어", label="번역 언어")

    with gr.Row():
        input_text = gr.Textbox(lines=4, label="원본 텍스트")
        output_text = gr.Textbox(lines=4, label="번역 결과")
        output_audio = gr.Audio(label="음성 재생 (영어일 경우만)", type="filepath")

    translate_btn = gr.Button("번역하기")

    translate_btn.click(
        fn=translate_and_speak,
        inputs=[input_text, src_lang, tgt_lang],
        outputs=[output_text, output_audio]
    )

demo.launch()


C:\Users\Admin\miniforge3\envs\ai_serving\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--microsoft--speecht5_tts. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\Admin\miniforge3\envs\ai_serving\Lib\site-packages\huggingface_hub\file_download.py:143: UserWa

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
